# AdDhakhiraCorpusAI - Assistant Bibliographique Web App

Configuration `default` : `gemma-4-12B-it` pour l'extraction, `Qwen3.6-35B-A3B` pour le raisonnement, retrieval dense FAISS avec `Qwen/Qwen3-Embedding-4B`.
La liste déroulante propose aussi `lite_version`, Gemini, ChatGPT/OpenAI ou Claude/Anthropic et remplacent uniquement les deux LLM ; le retrieval, les pages et la sortie restent identiques.

## Utilisation
1. Clique sur le bouton **Play** de la cellule `Initialiser`.
2. Attends le message `Initialisation terminee`.
3. Clique sur le bouton **Play** de la cellule `Ouvrir l'interface`.
4. Choisis le backend d'inférence, écris ta question puis clique sur **Envoyer**.


## Etape 1 - Initialiser
Clique sur **Play**. Cette étape installe AdDhakhira sur Google Drive, prépare les modèles locaux si demandé et construit l'index dense FAISS si nécessaire.
Tu peux changer les modèles via `MODEL_EXTRACTOR_ID`, `MODEL_REASONER_ID` et `EMBEDDING_MODEL_ID`.


In [ ]:
#@title Etape 1 - Initialiser
CONFIGURATION = "lite_version" #@param ["default", "lite_version", "gemini_api", "openai_api", "anthropic_api"]
#@markdown ---
#@markdown **Paramètres généraux**
REPO_GIT_URL = "https://github.com/git-haddadz/AdDhakhiraCorpusAI.git" #@param {type:"string"}
REPO_DIR = "/content/drive/MyDrive/AdDhakhiraCorpusAI/repo" #@param {type:"string"}
DRIVE_DIR = "/content/drive/MyDrive/AdDhakhiraCorpusAI" #@param {type:"string"}
#@markdown ---

import os
import sys
import subprocess
import hashlib
from pathlib import Path
from google.colab import drive

# Valeurs par défaut affichées ensuite dans Gradio selon CONFIGURATION.
MODEL_EXTRACTOR_ID = "gemma-4-12B-it"
MODEL_REASONER_ID = "Qwen3.6-35B-A3B"
NUM_GPUS_EXTRACTOR = 1
NUM_GPUS_REASONER = 1
LITE_MODEL_ID = "Qwen/Qwen2.5-7B-Instruct-AWQ"
GEMINI_API_KEY = ""
OPENAI_API_KEY = ""
ANTHROPIC_API_KEY = ""
GEMINI_MODEL = "gemini-2.5-flash"
OPENAI_MODEL = "gpt-4.1"
ANTHROPIC_MODEL = "claude-sonnet-4-20250514"
EMBEDDING_MODEL_ID = "Qwen/Qwen3-Embedding-4B" #@param {type:"string"}
ENABLE_DENSE_RETRIEVAL = True #@param {type:"boolean"}
USE_PREBUILT_DENSE_INDEX = True #@param {type:"boolean"}
PREBUILT_INDEX_REPO_ID = "userzh92/addhakhira-faiss-index" #@param {type:"string"}
PREBUILT_INDEX_PATH = "Qwen__Qwen3-Embedding-4B" #@param {type:"string"}
MODEL_STORAGE = "drive" #@param ["drive", "colab_session"]
FORCE_DOWNLOAD_MODELS = False

VALID_CONFIGURATIONS = {"default", "lite_version", "gemini_api", "openai_api", "anthropic_api"}
if CONFIGURATION not in VALID_CONFIGURATIONS:
    raise ValueError(f"CONFIGURATION invalide: {CONFIGURATION!r}")
ASSET_EVENTS = []


def run(cmd, cwd=None):
    print('$', ' '.join(map(str, cmd)), flush=True)
    subprocess.run(list(map(str, cmd)), check=True, cwd=cwd)


def run_output(cmd, cwd=None):
    print('$', ' '.join(map(str, cmd)), flush=True)
    return subprocess.run(list(map(str, cmd)), check=True, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True).stdout.strip()


def local_model_dir(root: Path, model_id: str) -> Path:
    return root / model_id.strip('/').replace('/', '__')


def model_snapshot_ready(model_dir: Path) -> bool:
    if not model_dir.is_dir():
        return False
    if not (model_dir / 'config.json').is_file():
        return False
    weight_patterns = ('*.safetensors', '*.bin', '*.pt', '*.gguf')
    has_weights = any(next(model_dir.glob(pattern), None) is not None for pattern in weight_patterns)
    has_sharded_weights = (model_dir / 'model.safetensors.index.json').is_file() or (model_dir / 'pytorch_model.bin.index.json').is_file()
    return has_weights or has_sharded_weights


def model_ref(model_dir: Path, model_id: str) -> str:
    return str(model_dir) if model_snapshot_ready(model_dir) else model_id


def local_model_path(model_id: str, root: Path = None) -> Path:
    root = root or models_dir
    return root / model_id.strip('/').replace('/', '__')


def ensure_model_available(model_id: str, force: bool = False, storage_root: Path = None, storage_label: str = None) -> str:
    model_id = (model_id or '').strip()
    if not model_id:
        raise ValueError('Model id/path vide.')
    candidate = Path(model_id)
    if candidate.exists():
        print('Modele local trouve:', candidate)
        ASSET_EVENTS.append(f'Modele local trouve: {candidate}')
        return str(candidate)
    root = storage_root or models_dir
    label = storage_label or ('Colab session' if root == session_models_dir else 'Drive')
    target_dir = local_model_path(model_id, root=root)
    if model_snapshot_ready(target_dir) and not force:
        print(f'Modele deja present dans {label}:', target_dir)
        ASSET_EVENTS.append(f'Modele deja present dans {label}: {target_dir}')
        return str(target_dir)
    if target_dir.exists() and not model_snapshot_ready(target_dir):
        print(f'Modele incomplet dans {label}, telechargement/reprise:', target_dir)
        ASSET_EVENTS.append(f'Modele incomplet, reprise du telechargement: {target_dir}')
    print(f'Telechargement du modele vers {label}:', model_id, '->', target_dir)
    ASSET_EVENTS.append(f'Telechargement du modele: {model_id}')
    from huggingface_hub import snapshot_download
    token = os.environ.get('HF_TOKEN', None)
    snapshot_download(repo_id=model_id, local_dir=str(target_dir), local_dir_use_symlinks=False, token=token)
    if not model_snapshot_ready(target_dir):
        raise RuntimeError(f'Telechargement incomplet ou invalide pour {model_id}: {target_dir}')
    return str(target_dir)


if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')
else:
    print('Drive deja monte.')

project_dir = Path(DRIVE_DIR)
drive_models_dir = project_dir / 'models'
session_project_dir = Path('/content/AdDhakhiraCorpusAI_session')
session_models_dir = session_project_dir / 'models'
models_dir = drive_models_dir if MODEL_STORAGE == 'drive' else session_models_dir
index_dir = project_dir / 'vector_indexes'
output_dir = project_dir / 'outputs'
drive_hf_cache_dir = project_dir / 'hf_cache'
session_hf_cache_dir = session_project_dir / 'hf_cache'
hf_cache_dir = drive_hf_cache_dir if MODEL_STORAGE == 'drive' else session_hf_cache_dir
drive_models_dir.mkdir(parents=True, exist_ok=True)
session_models_dir.mkdir(parents=True, exist_ok=True)
index_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)
hf_cache_dir.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(hf_cache_dir)
os.environ['TRANSFORMERS_CACHE'] = str(hf_cache_dir / 'transformers')
os.environ['SENTENCE_TRANSFORMERS_HOME'] = str(hf_cache_dir / 'sentence_transformers')

repo_path = Path(REPO_DIR)
if not (repo_path / '.git').exists():
    if repo_path.exists() and any(repo_path.iterdir()):
        raise RuntimeError(f'REPO_DIR existe mais ne contient pas un clone git valide: {repo_path}')
    repo_path.parent.mkdir(parents=True, exist_ok=True)
    run(['git', 'clone', REPO_GIT_URL, str(repo_path)])
else:
    run(['git', 'fetch', 'origin', 'main'], cwd=repo_path)
    local_rev = run_output(['git', 'rev-parse', 'HEAD'], cwd=repo_path)
    remote_rev = run_output(['git', 'rev-parse', 'origin/main'], cwd=repo_path)
    if local_rev != remote_rev:
        run(['git', 'pull', '--ff-only', 'origin', 'main'], cwd=repo_path)
    else:
        print('Repo deja a jour.')
os.chdir(repo_path)

requirements_path = repo_path / 'requirements.txt'
requirements_hash = hashlib.sha256(requirements_path.read_bytes()).hexdigest()[:12]
gpu_name = subprocess.run(
    ['bash', '-lc', 'nvidia-smi --query-gpu=name --format=csv,noheader 2>/dev/null | head -n 1'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
).stdout.strip().replace(' ', '_') or 'no_gpu'

deps_marker = Path('/content') / f'.colab_deps_ok_{requirements_hash}_{gpu_name}'

if not deps_marker.exists():
    run([sys.executable, '-m', 'pip', 'install', '-U', 'pip'])
    run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'])
    deps_marker.write_text('ok', encoding='utf-8')
    raise RuntimeError('Dependances installees. Redemarre le runtime Colab, puis relance cette cellule.')

INITIAL_BACKEND = CONFIGURATION
BASE_CONFIG_DEFAULTS = {
    'JSON_INPUT_PATH': str(repo_path / 'database'),
    'TOP_K_CHUNKS': 8,
    'TOP_K_PAGES': 5,
    'MAX_MODEL_LEN_EXTRACTOR': 4096,
    'MIN_MODEL_LEN_REASONER': 8192,
    'VLLM_GPU_MEMORY_UTILIZATION': 0.90,
    'VLLM_MAX_NUM_BATCHED_TOKENS': None,
    'VECTOR_INDEX_BACKEND': 'faiss',
    'DENSE_TOP_K': 40,
    'HYBRID_DENSE_WEIGHT': 0.40,
    'HYBRID_LEXICAL_WEIGHT': 0.60,
    'ENABLE_HYBRID_RETRIEVAL': False,
    'TRANSLATE_TOP_CHUNKS_TO_FRENCH': False,
    'TRANSLATION_MAX_TOKENS': 4096,
    'AUTO_TRANSLATE_QUESTION_TO_ARABIC': False,
    'REASONER_TEMPERATURE': 0.2,
    'REASONER_TOP_P': 0.9,
    'REASONER_OUTPUT_MAX_TOKENS': 1200,
    'REASONER_CONTEXT_SAFETY_TOKENS': 8000,
    'JSON_GENERATION_MAX_RETRIES': 4,
    'JSON_GENERATION_MAX_TOKEN_MULTIPLIER': 4,
}


def embedding_model_refs_equivalent(left: str, right: str) -> bool:
    left = str(left or '').rstrip('/')
    right = str(right or '').rstrip('/')
    if left == right:
        return True
    left_name = Path(left).name
    right_name = Path(right).name
    if left_name and right_name and left_name == right_name:
        return True
    return left.replace('/', '__') == right.replace('/', '__')


def dense_index_is_compatible(embedding_model: str) -> bool:
    from src.data_loader import load_chunks
    from src.vector_index import build_signature, index_dir_for

    directory = index_dir_for(index_dir, embedding_model, 'faiss')
    sig_path = directory / 'signature.json'
    metadata_path = directory / 'metadata.json'
    index_path = directory / 'index.faiss'
    if not (sig_path.is_file() and metadata_path.is_file() and index_path.is_file()):
        return False
    chunks, _ = load_chunks(str(repo_path / 'database'))
    expected = build_signature(embedding_model, 'faiss', chunks)
    import json
    existing = json.loads(sig_path.read_text(encoding='utf-8'))
    strict_keys = ('index_version', 'backend', 'chunks_sha256', 'chunk_count')
    if not all(existing.get(k) == expected.get(k) for k in strict_keys):
        return False
    existing_model = str(existing.get('embedding_model', ''))
    expected_model = str(expected.get('embedding_model', ''))
    if existing_model.rstrip('/') == expected_model.rstrip('/'):
        return True
    if embedding_model_refs_equivalent(existing_model, expected_model):
        existing['embedding_model'] = expected['embedding_model']
        sig_path.write_text(json.dumps(existing, ensure_ascii=False, indent=2), encoding='utf-8')
        print('Signature index normalisee pour le chemin embedding courant.')
        return True
    return False


def download_prebuilt_dense_index(embedding_model: str) -> bool:
    if not USE_PREBUILT_DENSE_INDEX:
        return False
    from huggingface_hub import hf_hub_download
    from src.vector_index import index_dir_for

    target_dir = index_dir_for(index_dir, embedding_model, 'faiss')
    target_dir.mkdir(parents=True, exist_ok=True)
    token = os.environ.get('HF_TOKEN', None)
    print('Telechargement de l\'index dense preconstruit depuis Hugging Face:', PREBUILT_INDEX_REPO_ID)
    ASSET_EVENTS.append(f'Telechargement index dense preconstruit: {PREBUILT_INDEX_REPO_ID}/{PREBUILT_INDEX_PATH}')
    for filename in ('signature.json', 'metadata.json', 'index.faiss'):
        downloaded = hf_hub_download(
            repo_id=PREBUILT_INDEX_REPO_ID,
            repo_type='dataset',
            filename=f'{PREBUILT_INDEX_PATH}/{filename}',
            local_dir=str(target_dir.parent),
            token=token,
        )
        source = Path(downloaded)
        destination = target_dir / filename
        if source.resolve() != destination.resolve():
            destination.write_bytes(source.read_bytes())
        print('Index file pret:', destination)
    return dense_index_is_compatible(embedding_model)


def ensure_dense_index_available(embedding_model: str):
    if dense_index_is_compatible(embedding_model):
        print('Index dense compatible deja present.')
        ASSET_EVENTS.append(f'Index dense compatible deja present: {index_dir}')
        return
    if download_prebuilt_dense_index(embedding_model):
        print('Index dense preconstruit telecharge et compatible.')
        ASSET_EVENTS.append(f'Index dense preconstruit compatible: {index_dir}')
        return
    from src.vector_index import build_and_save_index
    print('Verification/construction de l\'index dense...')
    ASSET_EVENTS.append(f'Verification/construction de l\'index dense: {index_dir}')
    build_and_save_index(
        json_input_path=repo_path / 'database',
        output_dir=index_dir,
        model_name=embedding_model,
        backend='faiss',
        batch_size=32,
        force=False,
        show_progress=True,
    )
    ASSET_EVENTS.append(f'Index dense pret: {index_dir}')


def prepare_initial_assets():
    if ENABLE_DENSE_RETRIEVAL:
        embedding_model = ensure_model_available(EMBEDDING_MODEL_ID, force=FORCE_DOWNLOAD_MODELS)
        ensure_dense_index_available(embedding_model)
    if CONFIGURATION == 'lite_version':
        ensure_model_available(LITE_MODEL_ID, force=FORCE_DOWNLOAD_MODELS)
    elif CONFIGURATION == 'default':
        ensure_model_available(MODEL_EXTRACTOR_ID, force=FORCE_DOWNLOAD_MODELS)
        ensure_model_available(MODEL_REASONER_ID, force=FORCE_DOWNLOAD_MODELS)


prepare_initial_assets()
ASSET_STATUS = '\n'.join(f'- {event}' for event in ASSET_EVENTS) or '- Aucun artefact prepare.'

print('Initialisation terminee')
print('Configuration initiale:', INITIAL_BACKEND)
print('Repo:', repo_path)
print('Model storage:', MODEL_STORAGE)
print('Models dir:', models_dir)
print('Drive models dir:', drive_models_dir)
print('Vector index:', index_dir)
print('Outputs:', output_dir)

## Etape 2 - Ouvrir l'interface
Clique sur **Play**. Une interface de chat apparaît juste en dessous.


In [ ]:
#@title Etape 2 - Ouvrir l'interface
import ast
import os
import sys
import time
from datetime import datetime
from pathlib import Path

import gradio as gr

drive_dir = Path(DRIVE_DIR)
session_dir = drive_dir / 'outputs' / f"session_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
session_dir.mkdir(parents=True, exist_ok=True)
counter = {'n': 0}


def load_existing_config_api_keys():
    config_path = Path(REPO_DIR) / 'src' / 'config.py'
    if not config_path.exists():
        return {}
    keys = {}
    tree = ast.parse(config_path.read_text(encoding='utf-8'))
    wanted = {'GEMINI_API_KEY', 'OPENAI_API_KEY', 'ANTHROPIC_API_KEY'}
    for node in tree.body:
        if not isinstance(node, ast.Assign) or len(node.targets) != 1:
            continue
        target = node.targets[0]
        if isinstance(target, ast.Name) and target.id in wanted:
            try:
                value = ast.literal_eval(node.value)
            except Exception:
                value = ''
            if isinstance(value, str) and value:
                keys[target.id] = value
    return keys


existing_api_keys = load_existing_config_api_keys()
GEMINI_API_KEY = GEMINI_API_KEY or existing_api_keys.get('GEMINI_API_KEY', '')
OPENAI_API_KEY = OPENAI_API_KEY or existing_api_keys.get('OPENAI_API_KEY', '')
ANTHROPIC_API_KEY = ANTHROPIC_API_KEY or existing_api_keys.get('ANTHROPIC_API_KEY', '')


def write_config_file(cfg):
    lines = [
        'from pathlib import Path',
        '',
        '# This file was generated by the notebook UI.',
        '# For manual local usage, copy src/config_template.py to src/config.py and edit it.',
        '',
    ]
    order = [
        'LLM_BACKEND', 'GEMINI_API_KEY', 'OPENAI_API_KEY', 'ANTHROPIC_API_KEY',
        'MODEL_EXTRACTOR_PATH', 'MODEL_REASONER_PATH', 'JSON_INPUT_PATH',
        'TOP_K_CHUNKS', 'TOP_K_PAGES', 'MAX_MODEL_LEN_EXTRACTOR', 'MIN_MODEL_LEN_REASONER',
        'NUM_GPUS_EXTRACTOR', 'NUM_GPUS_REASONER', 'VLLM_GPU_MEMORY_UTILIZATION',
        'VLLM_MAX_NUM_BATCHED_TOKENS', 'EMBEDDING_MODEL', 'EMBEDDING_INDEX_DIR',
        'VECTOR_INDEX_BACKEND', 'DENSE_TOP_K', 'HYBRID_DENSE_WEIGHT', 'HYBRID_LEXICAL_WEIGHT',
        'ENABLE_DENSE_RETRIEVAL', 'ENABLE_HYBRID_RETRIEVAL', 'TRANSLATE_TOP_CHUNKS_TO_FRENCH',
        'TRANSLATION_MAX_TOKENS', 'AUTO_TRANSLATE_QUESTION_TO_ARABIC', 'REASONER_TEMPERATURE',
        'REASONER_TOP_P', 'REASONER_OUTPUT_MAX_TOKENS', 'REASONER_CONTEXT_SAFETY_TOKENS',
        'JSON_GENERATION_MAX_RETRIES', 'JSON_GENERATION_MAX_TOKEN_MULTIPLIER',
    ]
    for key in order:
        value = cfg[key]
        if key in {'JSON_INPUT_PATH', 'EMBEDDING_INDEX_DIR'}:
            lines.append(f'{key} = Path({str(value)!r})')
        else:
            lines.append(f'{key} = {value!r}')
    config_path = Path(REPO_DIR) / 'src' / 'config.py'
    config_path.write_text('\n'.join(lines) + '\n', encoding='utf-8')


def clear_pipeline_modules():
    for name in list(sys.modules):
        if name == 'src.config' or name.startswith(('src.pipeline', 'src.llm_ops', 'src.llm_backend', 'src.retrieval')):
            sys.modules.pop(name, None)


def resolve_model(model_id):
    model_id = (model_id or '').strip()
    if not model_id:
        raise ValueError('Model id/path vide.')
    return ensure_model_available(model_id, force=False)


def build_backend_config(
    backend,
    model_extractor_id,
    model_reasoner_id,
    num_gpus_extractor,
    num_gpus_reasoner,
    lite_model_id,
    gemini_api_key,
    openai_api_key,
    anthropic_api_key,
    gemini_model,
    openai_model,
    anthropic_model,
    use_dense_retrieval,
):
    cfg = dict(BASE_CONFIG_DEFAULTS)
    dense_enabled = bool(use_dense_retrieval and ENABLE_DENSE_RETRIEVAL)
    embedding_model = ensure_model_available(EMBEDDING_MODEL_ID, force=False) if dense_enabled else model_ref(local_model_path(EMBEDDING_MODEL_ID), EMBEDDING_MODEL_ID)
    cfg.update({
        'NUM_GPUS_EXTRACTOR': int(num_gpus_extractor),
        'NUM_GPUS_REASONER': int(num_gpus_reasoner),
        'EMBEDDING_MODEL': embedding_model,
        'EMBEDDING_INDEX_DIR': str(index_dir),
        'ENABLE_DENSE_RETRIEVAL': dense_enabled,
    })
    if backend == 'gemini_api':
        if not (gemini_api_key or '').strip():
            raise ValueError('GEMINI_API_KEY est requis pour le backend Gemini.')
        cfg.update({
            'LLM_BACKEND': 'gemini_api',
            'MODEL_EXTRACTOR_PATH': gemini_model.strip(),
            'MODEL_REASONER_PATH': gemini_model.strip(),
            'GEMINI_API_KEY': gemini_api_key.strip(),
            'OPENAI_API_KEY': '',
            'ANTHROPIC_API_KEY': '',
        })
    elif backend == 'openai_api':
        if not (openai_api_key or '').strip():
            raise ValueError('OPENAI_API_KEY est requis pour le backend OpenAI.')
        cfg.update({
            'LLM_BACKEND': 'openai_api',
            'MODEL_EXTRACTOR_PATH': openai_model.strip(),
            'MODEL_REASONER_PATH': openai_model.strip(),
            'GEMINI_API_KEY': '',
            'OPENAI_API_KEY': openai_api_key.strip(),
            'ANTHROPIC_API_KEY': '',
        })
    elif backend == 'anthropic_api':
        if not (anthropic_api_key or '').strip():
            raise ValueError('ANTHROPIC_API_KEY est requis pour le backend Anthropic.')
        cfg.update({
            'LLM_BACKEND': 'anthropic_api',
            'MODEL_EXTRACTOR_PATH': anthropic_model.strip(),
            'MODEL_REASONER_PATH': anthropic_model.strip(),
            'GEMINI_API_KEY': '',
            'OPENAI_API_KEY': '',
            'ANTHROPIC_API_KEY': anthropic_api_key.strip(),
        })
    elif backend == 'lite_version':
        lite_model = resolve_model(lite_model_id)
        cfg.update({
            'LLM_BACKEND': 'default',
            'MODEL_EXTRACTOR_PATH': lite_model,
            'MODEL_REASONER_PATH': lite_model,
            'GEMINI_API_KEY': '',
            'OPENAI_API_KEY': '',
            'ANTHROPIC_API_KEY': '',
        })
    else:
        extractor_model = resolve_model(model_extractor_id)
        reasoner_model = resolve_model(model_reasoner_id)
        cfg.update({
            'LLM_BACKEND': 'default',
            'MODEL_EXTRACTOR_PATH': extractor_model,
            'MODEL_REASONER_PATH': reasoner_model,
            'GEMINI_API_KEY': '',
            'OPENAI_API_KEY': '',
            'ANTHROPIC_API_KEY': '',
        })
    return cfg


def ask(*args):
    (
        backend,
        model_extractor_id,
        model_reasoner_id,
        num_gpus_extractor,
        num_gpus_reasoner,
        lite_model_id,
        gemini_api_key,
        openai_api_key,
        anthropic_api_key,
        gemini_model,
        openai_model,
        anthropic_model,
        use_dense_retrieval,
        question,
    ) = args
    question = (question or '').strip()
    if not question:
        return 'Saisis une question.', '', None
    try:
        cfg = build_backend_config(
            backend,
            model_extractor_id,
            model_reasoner_id,
            num_gpus_extractor,
            num_gpus_reasoner,
            lite_model_id,
            gemini_api_key,
            openai_api_key,
            anthropic_api_key,
            gemini_model,
            openai_model,
            anthropic_model,
            use_dense_retrieval,
        )
        write_config_file(cfg)
        clear_pipeline_modules()
        from src.pipeline import build_final_report
        from src.reporting import write_output_with_timing

        counter['n'] += 1
        out_path = session_dir / f'output_{counter["n"]}.html'
        t0 = time.time()
        report = build_final_report(question=question, translate_to_french=False, diagnostic_coherence=True)
        elapsed = time.time() - t0
        write_output_with_timing(report, elapsed_seconds=elapsed, output_path=str(out_path))
        status = f'Termine en {elapsed:.1f}s avec {backend}. HTML sauvegarde : {out_path}'
        return status, report, str(out_path)
    except Exception as exc:
        return f'Erreur: {exc}', '', None


def toggle_config_fields(backend):
    return (
        gr.update(visible=(backend == 'default')),
        gr.update(visible=(backend == 'lite_version')),
        gr.update(visible=(backend == 'gemini_api')),
        gr.update(visible=(backend == 'openai_api')),
        gr.update(visible=(backend == 'anthropic_api')),
    )


with gr.Blocks(title='AdDhakhiraCorpusAI') as demo:
    gr.Markdown('## Assistant de recherche Malikite')
    gr.Markdown('### Artefacts prepares\n' + ASSET_STATUS)
    backend = gr.Dropdown(
        choices=[('Default local vLLM', 'default'), ('Lite version', 'lite_version'), ('Gemini API', 'gemini_api'), ('ChatGPT / OpenAI API', 'openai_api'), ('Claude / Anthropic API', 'anthropic_api')],
        value=INITIAL_BACKEND,
        label='Configuration',
    )
    with gr.Group(visible=(INITIAL_BACKEND == 'default')) as default_group:
        model_extractor_id = gr.Textbox(label='MODEL_EXTRACTOR_ID', value=MODEL_EXTRACTOR_ID)
        model_reasoner_id = gr.Textbox(label='MODEL_REASONER_ID', value=MODEL_REASONER_ID)
        with gr.Row():
            num_gpus_extractor = gr.Number(label='NUM_GPUS_EXTRACTOR', value=NUM_GPUS_EXTRACTOR, precision=0)
            num_gpus_reasoner = gr.Number(label='NUM_GPUS_REASONER', value=NUM_GPUS_REASONER, precision=0)
    with gr.Group(visible=(INITIAL_BACKEND == 'lite_version')) as lite_group:
        lite_model_id = gr.Textbox(label='LITE_MODEL_ID', value=LITE_MODEL_ID)
    with gr.Group(visible=(INITIAL_BACKEND == 'gemini_api')) as gemini_group:
        gemini_api_key = gr.Textbox(label='GEMINI_API_KEY', type='password', value=GEMINI_API_KEY)
        gemini_model = gr.Textbox(label='GEMINI_MODEL', value=GEMINI_MODEL)
    with gr.Group(visible=(INITIAL_BACKEND == 'openai_api')) as openai_group:
        openai_api_key = gr.Textbox(label='OPENAI_API_KEY', type='password', value=OPENAI_API_KEY)
        openai_model = gr.Textbox(label='OPENAI_MODEL', value=OPENAI_MODEL)
    with gr.Group(visible=(INITIAL_BACKEND == 'anthropic_api')) as anthropic_group:
        anthropic_api_key = gr.Textbox(label='ANTHROPIC_API_KEY', type='password', value=ANTHROPIC_API_KEY)
        anthropic_model = gr.Textbox(label='ANTHROPIC_MODEL', value=ANTHROPIC_MODEL)
    with gr.Accordion('Retrieval prepare', open=False):
        gr.Textbox(label='EMBEDDING_MODEL_ID', value=EMBEDDING_MODEL_ID, interactive=False)
        use_dense_retrieval = gr.Checkbox(label='USE_DENSE_RETRIEVAL', value=ENABLE_DENSE_RETRIEVAL, interactive=ENABLE_DENSE_RETRIEVAL)
    question = gr.Textbox(label='Question', lines=4)
    submit = gr.Button('Envoyer', variant='primary')
    status = gr.Markdown()
    answer = gr.HTML()
    download = gr.File(label='Telecharger la reponse HTML')

    backend.change(
        toggle_config_fields,
        inputs=backend,
        outputs=[default_group, lite_group, gemini_group, openai_group, anthropic_group],
    )
    submit.click(
        ask,
        inputs=[
            backend,
            model_extractor_id,
            model_reasoner_id,
            num_gpus_extractor,
            num_gpus_reasoner,
            lite_model_id,
            gemini_api_key,
            openai_api_key,
            anthropic_api_key,
            gemini_model,
            openai_model,
            anthropic_model,
            use_dense_retrieval,
            question,
        ],
        outputs=[status, answer, download],
    )

print('Dossier de sortie:', session_dir)
demo.launch(share=True, debug=False)
